In [20]:
import pyreadr
import pandas as pd
import re
import xml
from bs4 import BeautifulSoup
import nltk
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, recall_score, classification_report, confusion_matrix
import numpy as np

In [21]:
raw_claims_df = pyreadr.read_r("../../data/claims-raw.RData")["claims_raw"]
raw_claims_df.shape

(2165, 9)

In [26]:
# Code from task 1 to reproduce same cleaned dataset
def parse_fn(html, selector = None):
    soup = BeautifulSoup(html, features = "lxml")
    try:
        for script in soup(["script", "style"]):
            script.decompose() # destroy script and style elements

        if selector is None:
            raw = ' '.join(spoon.get_text(' '))
        else:
            raw = ' '.join(spoon.get_text(' ') for spoon in soup.select(selector))
        
        # remove urls and emails
            # \S@\S+ means non-whitespace characters followed by @ then followed by 1 or more non-whitespace characters
            # pattern = r'https?://\S+|www\.\S+' which means http with optional s followed by :// and then 1 or more non-whitespace characters
            # |www\.\S+ means or www. then 1 or more non-whitespace characters
        text = re.sub(r'\S+@\S+|https?://\S+|www\.\S+', "", raw)

        # remove apostraphes

        text = re.sub("'", "",text)

        # remove non-alphabetic characters and nbsp
        text = re.sub(r'[^a-zA-Z\s]', ' ', text)

        text = re.sub(r'([a-z])([A-Z])', r'\1 \2', text) # add space between camel case words

        text = text.lower()

        text = re.sub(r'\s+', ' ', text).strip() # collapse multiple spaces into one and strip leading/trailing whitespace
        # remove punctuation and newlines
        
        return text 
    except Exception as err:
        print(f"Error parsing HTML: {err}")
        return None


def parse_data(df, selector = None):
    df = df[df['text_tmp'].str.contains('<!', regex = False)].copy() # filter for rows that contain html tags
    df['text_clean'] = df['text_tmp'].apply(parse_fn, selector = selector) # apply parse_fn and create a new column
    return df

In [27]:
clean_claims_df = parse_data(raw_claims_df, selector = 'h1, h2, h3, h4, h5, h6, p')

C:\Users\posit\AppData\Local\Temp\ipykernel_8428\3430768308.py:3: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html, features = "lxml")


In [29]:
# Match task 1 split
# same row order, seed, proportion, and no stratification.
train_df, test_df = train_test_split(
    clean_claims_df,
    test_size=0.2,
    random_state=101422
)

# Use only training pages for architecture experiments.
fit_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df["bclass"]
)

assert clean_claims_df[".id"].is_unique
assert clean_claims_df["text_clean"].notna().all()

print("Full training:", len(train_df))
print("Fitting:", len(fit_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Full training: 1712
Fitting: 1369
Validation: 343
Test: 428
